# Predicting serious food-safety violations in Seattle restaurants

**Summary**

The asian and ethnic flags look like the strongest predictor in this data, but I don't think they can be trusted. The scores for Asian and other ethnic restaurants don't behave like real inspection records (section 2.2), so most of the model's accuracy comes from that label problem. For the other restaurants, where the records look fine, the best model gets an AUC of 0.66, only a bit better than ranking by the last inspection score (0.64). Inspection history does almost all the work and Yelp adds nothing. Used as it is, the model would send every priority inspection to Asian and ethnic restaurants, which is the main thing I'd want a health department to know before using something like this.

In [1]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.precision", 3)

## 0. Load/check the data

Each row is one inspection, not one restaurant. It has the inspection's penalty score plus what was known before it: the previous score, an average of earlier scores, and the Yelp reviews since the last inspection.

In [2]:
df = pd.read_csv("data/food_violations_seattle.csv", index_col=0,
                 parse_dates=["inspection_period_start_date", "inspection_period_end_date"])
lookup = pd.read_csv("data/cuisines.csv", index_col="x")

# shorter column names
df = df.rename(columns={
    "inspection_period_start_date": "period_start",
    "inspection_period_end_date": "inspection_date",
    "inspection_penalty_score": "score",
    "inspection_prev_penalty_score": "prev_score",
    "inspection_average_prev_penalty_scores": "avg_prev_score",
    "average_review_rating": "avg_rating",
    "review_contents": "review_text",
})

# quick sanity checks
assert df.inspection_id.is_unique
assert df.notna().all().all()
assert (df.period_start < df.inspection_date).all()

print(len(df), "inspections,", df.restaurant_id.nunique(), "restaurants,",
      df.inspection_date.min().date(), "to", df.inspection_date.max().date())
df.drop(columns="review_text").head(3)

13299 inspections, 1756 restaurants, 2006-01-03 to 2013-02-08


,inspection_id,restaurant_id,period_start,inspection_date,score,avg_prev_score,prev_score,cuisines,zip_code,review_count,non_positive_review_count,avg_rating
0,DA2383404,ZriNDCVxkCdVEO-X2sjHFw,2012-02-06,2012-10-02,46,54.0,50,"['Japanese', 'Restaurants']",98116,1,0,5.000
1,DA2304227,ZriNDCVxkCdVEO-X2sjHFw,2011-11-03,2012-02-06,82,86.0,76,"['Japanese', 'Restaurants']",98116,3,0,4.000
2,DA2278690,ZriNDCVxkCdVEO-X2sjHFw,2011-01-06,2011-11-03,24,67.0,41,"['Japanese', 'Restaurants']",98116,7,0,4.429


Two rows have a -1 placeholder. One inspection scored -1 and the next one at the same restaurant carries it as its previous score. Scores can't be negative, so I drop both.

In [3]:
bad = (df[["score", "prev_score", "avg_prev_score"]] < 0).any(axis=1)
display(df.loc[bad, ["restaurant_id", "inspection_date", "score", "prev_score", "avg_prev_score"]])

df = df[~bad].copy()
df["serious"] = (df.score >= 10).astype(int)
print(len(df), "inspections left")
print(df.duplicated(["restaurant_id", "inspection_date"], keep=False).sum(),
      "rows share a restaurant and date with another row (different inspection IDs, so I keep them)")

,restaurant_id,inspection_date,score,prev_score,avg_prev_score
9049,DRpffXwLMlMqKN_QX2a0vw,2012-11-27,0,-1,-1.0
9050,DRpffXwLMlMqKN_QX2a0vw,2012-04-10,-1,0,0.0


13297 inspections left
171 rows share a restaurant and date with another row (different inspection IDs, so I keep them)
